In [4]:
import socket
print(socket.gethostname())

awr-2-04


In [7]:
import warnings

import xarray as xr
import hdf5plugin
import matplotlib.pyplot as plt

In [3]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=xr.SerializationWarning)
    dsp = xr.open_mfdataset('/cw3e/mead/projects/cwp167/moerfani_data/regional/2019/01P/wwrf_reanalysis_modellev_d01_2019-01-01*.nc', engine='h5netcdf', data_vars='all')
    dss = xr.open_mfdataset('/cw3e/mead/projects/cwp167/moerfani_data/regional/2019/01S/wwrf_reanalysis_singlelev_d01_2019-01-01*.nc', engine='h5netcdf', data_vars='all')

We will extract the pressure variables and single level variables from all 24 nc files representing one day as follows. These variables will then be resampled to 6-hour intervals and stored in a new netcdf file. For the precipitation variable, this resampling should be different (e.g., accumulation over the period).

**Pressure variables:**
- Geopotential (Z_e)
- Temperature (T_e)
- Specific humidity (q_e)
- Zonal wind (u_gr_e)
- Meridional wind (v_gr_e)

**Single levels:**
- Mean sea level pressure (slp)
- Surface pressure (p_sfc)
- 2-m dewpoint temperature (Td_2m)
- 2-m temperature (T_2m)
- Skin/sea surface temperature (T_sfc)
- Integrated water vapor (IWV)
- Integrated vapor transport (meridional) (IVTV)
- Integrated vapor transport (zonal) (IVTU)
- 10-m zonal wind (u_10m_gr)
- 10-m meridional wind (v_10m_gr)
- 6-hourly accumulated precipitation (precip_bkt)

**Forcing:**
- Geopotential at the surface (Z_sfc)

In [4]:
dsp

<xarray.Dataset> Size: 167GB
Dimensions:            (time: 24, eta: 99, south_north: 970, west_east: 1389)
Coordinates:
  * time               (time) datetime64[ns] 192B 2019-01-01 ... 2019-01-01T2...
  * eta                (eta) float32 396B 0.998 0.9932 ... 0.001282 0.0004168
  * south_north        (south_north) float32 4kB -2.827e+03 ... 2.988e+03
  * west_east          (west_east) float32 6kB -5.238e+03 ... 3.091e+03
    lat                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
    lon                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
Data variables: (12/21)
    DateTime           (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
    T_e                (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    quantization_info  (time) |S1 24B b'' b'' b'' b'' b'' ... b'' b'' b'' b''
    Z_e                (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    Z_sfc              (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 970, 1389), meta=np.ndarray>
    day                (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                 ...
    r_rain             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    r_snow             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    u_gr_e             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    v_gr_e             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    w_e                (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    year               (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF 1.6, Standard Name Table v19
    source:         /cw3e/mead/projects/cnt117/reanalysis_temp/input/2018/d01...
    institution:    CW3E - Scripps Institution of Oceanography
    title:          /cw3e/mead/projects/cnt117/reanalysis_temp/output/2018/d0...
    notes:          Created with NCL script:  wrfout_to_cf.ncl v2.0
    created_by:     Daniel Steinhoff - dsteinhoff@ucsd.edu
    creation_date:  Fri Dec  5 01:18:13 PM PST 2025
    history:        Fri Dec  5 13:24:15 2025: /home/cw3eprod/miniforge3/envs/...
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....
    NCO:            netCDF Operators version 5.2.9 (Homepage = http://nco.sf....

In [ ]:
dss

<xarray.Dataset> Size: 9GB
Dimensions:            (time: 24, south_north: 970, west_east: 1389, soil: 4)
Coordinates:
  * time               (time) datetime64[ns] 192B 2019-01-01 ... 2019-01-01T2...
  * south_north        (south_north) float32 4kB -2.827e+03 ... 2.988e+03
  * west_east          (west_east) float32 6kB -5.238e+03 ... 3.091e+03
    lat                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
    lon                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
  * soil               (soil) float32 16B 0.04999 0.25 0.7002 1.5
Data variables: (12/74)
    AET                (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ALBEDO_ALL         (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 970, 1389), meta=np.ndarray>
    ALBEDO_BCK         (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ALBEDO_SFC         (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    CAPE               (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    CIN                (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ...                 ...
    v_150m_gr          (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    v_30m_gr           (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    v_80m_gr           (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    wd_10m             (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ws_10m             (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    year               (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF 1.6, Standard Name Table v19
    source:         /data/projects/40YearReanalysis/cwp103/2023.03.01_copy_fr...
    institution:    CW3E - Scripps Institution of Oceanography
    title:          /data/projects/40YearReanalysis/v2/output/singlelev/d01/2...
    notes:          Created with NCL script:  wrfout_to_cf.ncl v2.0
    created_by:     Daniel Steinhoff - dsteinhoff@ucsd.edu
    creation_date:  Tue Jan 20 20:10:24 PST 2026
    NCO:            netCDF Operators version 5.3.2 (Homepage = http://nco.sf....
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....

In [ ]:
# Extract pressure variables from dsp
pressure_vars = ['Z_e', 'T_e', 'q_e', 'u_gr_e', 'v_gr_e']
dsp_selected = dsp[pressure_vars]

# Extract single level and forcing variables from dss
single_vars = ['slp', 'p_sfc', 'Td_2m', 'T_2m', 'T_sfc', 'IWV', 'IVTV', 'IVTU', 'u_10m_gr', 'v_10m_gr', 'precip_bkt', 'Z_sfc']
dss_selected = dss[single_vars]

# Merge the datasets
ds_combined = xr.merge([dsp_selected, dss_selected])

# Resample to 6-hour intervals
# For precipitation, use sum (accumulation over the period)
precip_resampled = ds_combined['precip_bkt'].resample(time='6h').sum()

# For other variables, use mean
others = ds_combined.drop_vars('precip_bkt')
others_resampled = others.resample(time='6h').mean()

# Merge back the resampled data
ds_resampled = xr.merge([others_resampled, precip_resampled])

# Save to a new NetCDF file
save_path = '/cw3e/mead/projects/cwp167/moerfani_data/regional/2019/01M/wwrf_reanalysis_modellev_d01_2019-01-01.nc'
ds_resampled.to_netcdf(save_path)

In [8]:
save_path = '/cw3e/mead/projects/cwp167/moerfani_data/regional/2019/01M/wwrf_reanalysis_modellev_d01_2019-01-*.nc'
dsm = xr.open_mfdataset(save_path, engine='h5netcdf', data_vars='all')
dsm

<xarray.Dataset> Size: 339GB
Dimensions:      (time: 124, eta: 99, south_north: 970, west_east: 1389)
Coordinates:
  * time         (time) datetime64[ns] 992B 2019-01-01 ... 2019-01-31T18:00:00
  * eta          (eta) float32 396B 0.998 0.9932 0.9873 ... 0.001282 0.0004168
  * south_north  (south_north) float32 4kB -2.827e+03 -2.821e+03 ... 2.988e+03
  * west_east    (west_east) float32 6kB -5.238e+03 -5.232e+03 ... 3.091e+03
    lat          (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
    lon          (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
Data variables: (12/17)
    Z_e          (time, eta, south_north, west_east) float32 66GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    T_e          (time, eta, south_north, west_east) float32 66GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    q_e          (time, eta, south_north, west_east) float32 66GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    u_gr_e       (time, eta, south_north, west_east) float32 66GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    v_gr_e       (time, eta, south_north, west_east) float32 66GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    slp          (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    ...           ...
    IVTV         (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    IVTU         (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    u_10m_gr     (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    v_10m_gr     (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    Z_sfc        (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    precip_bkt   (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF 1.6, Standard Name Table v19
    source:         /cw3e/mead/projects/cnt117/reanalysis_temp/input/2018/d01...
    institution:    CW3E - Scripps Institution of Oceanography
    title:          /cw3e/mead/projects/cnt117/reanalysis_temp/output/2018/d0...
    notes:          Created with NCL script:  wrfout_to_cf.ncl v2.0
    created_by:     Daniel Steinhoff - dsteinhoff@ucsd.edu
    creation_date:  Fri Dec  5 01:18:13 PM PST 2025
    history:        Fri Dec  5 13:24:15 2025: /home/cw3eprod/miniforge3/envs/...
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....
    NCO:            netCDF Operators version 5.2.9 (Homepage = http://nco.sf....

To map your specific pressure levels to the provided eta ($\eta$) levels, we can calculate the closest mathematical match based on how terrain-following coordinates work.

Generally, the eta coordinate is defined as:
$\eta = \frac{p - p_t}{p_s - p_t}$

Where $p$ is the pressure at a given level, $p_s$ is the surface pressure, and $p_t$ is the model top pressure. If we assume a standard nominal surface pressure of **1013.25 hPa** and a top-of-atmosphere pressure of **0 hPa**, the target eta values simply become a fraction of the surface pressure ($p \approx \eta \times 1013.25$).

### The Nominal Mapping

By dividing your target pressure levels by 1013.25, we get the target $\eta$ values (e.g., 500 hPa becomes $\eta = 0.49346$). Finding the nearest neighbor in your provided array gives us this exact correspondence:

| Target Pressure (hPa) | Target $\eta$ | Closest Array $\eta$ | Array Index (0-indexed) | Nominal Pressure (hPa) |
| --- | --- | --- | --- | --- |
| **1000** | 0.98692 | 0.987305 | **2** | 1000.39 |
| **925** | 0.91290 | 0.916016 | **12** | 928.15 |
| **850** | 0.83888 | 0.840820 | **19** | 851.96 |
| **700** | 0.69084 | 0.690430 | **29** | 699.58 |
| **600** | 0.59215 | 0.600586 | **34** | 608.54 |
| **500** | 0.49346 | 0.486328 | **40** | 492.77 |
| **400** | 0.39477 | 0.392578 | **45** | 397.78 |
| **300** | 0.29609 | 0.291016 | **51** | 294.87 |
| **250** | 0.24673 | 0.246094 | **54** | 249.35 |
| **200** | 0.19738 | 0.193359 | **58** | 195.92 |
| **150** | 0.14804 | 0.149414 | **62** | 151.39 |
| **100** | 0.09869 | 0.097534 | **68** | 98.83 |
| **50** | 0.04935 | 0.049927 | **76** | 50.59 |

In [2]:
import numpy as np

# Your provided array of 99 eta levels
eta_levels = np.array([
    9.9804688e-01, 9.9316406e-01, 9.8730469e-01, 9.8242188e-01,
    9.7656250e-01, 9.6972656e-01, 9.6386719e-01, 9.5703125e-01,
    9.4921875e-01, 9.4140625e-01, 9.3359375e-01, 9.2480469e-01,
    9.1601562e-01, 9.0722656e-01, 8.9746094e-01, 8.8671875e-01,
    8.7597656e-01, 8.6425781e-01, 8.5253906e-01, 8.4082031e-01,
    8.2812500e-01, 8.1445312e-01, 8.0078125e-01, 7.8613281e-01,
    7.7148438e-01, 7.5585938e-01, 7.4023438e-01, 7.2460938e-01,
    7.0800781e-01, 6.9042969e-01, 6.7285156e-01, 6.5527344e-01,
    6.3769531e-01, 6.1914062e-01, 6.0058594e-01, 5.8203125e-01,
    5.6250000e-01, 5.4394531e-01, 5.2441406e-01, 5.0585938e-01,
    4.8632812e-01, 4.6777344e-01, 4.4824219e-01, 4.2968750e-01,
    4.1113281e-01, 3.9257812e-01, 3.7500000e-01, 3.5742188e-01,
    3.3984375e-01, 3.2324219e-01, 3.0664062e-01, 2.9101562e-01,
    2.7539062e-01, 2.6074219e-01, 2.4609375e-01, 2.3242188e-01,
    2.1875000e-01, 2.0605469e-01, 1.9335938e-01, 1.8164062e-01,
    1.7089844e-01, 1.6015625e-01, 1.4941406e-01, 1.3964844e-01,
    1.3085938e-01, 1.2207031e-01, 1.1328125e-01, 1.0546875e-01,
    9.7534180e-02, 9.0332031e-02, 8.3557129e-02, 7.7087402e-02,
    7.0983887e-02, 6.5246582e-02, 5.9875488e-02, 5.4748535e-02,
    4.9926758e-02, 4.5471191e-02, 4.1259766e-02, 3.7353516e-02,
    3.3691406e-02, 3.0395508e-02, 2.7282715e-02, 2.4414062e-02,
    2.1789551e-02, 1.9348145e-02, 1.7150879e-02, 1.5075684e-02,
    1.3122559e-02, 1.1352539e-02, 9.7274780e-03, 8.2168579e-03,
    6.8206787e-03, 5.5313110e-03, 4.3411255e-03, 3.2424927e-03,
    2.2201538e-03, 1.2817383e-03, 4.1675568e-04
])

# The exact target levels from the manuscript excerpt
target_pressures_hpa = np.array([50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000])

# Use the study's exact reference surface pressure assumption (1013.25 hPa)
# P_target / 1013.25 = Eta_target
target_etas = target_pressures_hpa / 1013.25

# Find the index of the nearest eta value for each target
nearest_indices = [np.abs(eta_levels - target).argmin() for target in target_etas]
matched_etas = eta_levels[nearest_indices]

# Print the correspondence mapping
print("Index Selection Based on 1013.25 hPa Reference Pressure:\n")
for p, eta, idx in zip(target_pressures_hpa, matched_etas, nearest_indices):
    nominal_p = eta * 1013.25
    print(f"Target: {p:>4} hPa -> Extract Index: {idx:>2} | (Eta: {eta:.6f}, Nominal P: {nominal_p:.2f} hPa)")

Index Selection Based on 1013.25 hPa Reference Pressure:

Target:   50 hPa -> Extract Index: 76 | (Eta: 0.049927, Nominal P: 50.59 hPa)
Target:  100 hPa -> Extract Index: 68 | (Eta: 0.097534, Nominal P: 98.83 hPa)
Target:  150 hPa -> Extract Index: 62 | (Eta: 0.149414, Nominal P: 151.39 hPa)
Target:  200 hPa -> Extract Index: 58 | (Eta: 0.193359, Nominal P: 195.92 hPa)
Target:  250 hPa -> Extract Index: 54 | (Eta: 0.246094, Nominal P: 249.35 hPa)
Target:  300 hPa -> Extract Index: 51 | (Eta: 0.291016, Nominal P: 294.87 hPa)
Target:  400 hPa -> Extract Index: 45 | (Eta: 0.392578, Nominal P: 397.78 hPa)
Target:  500 hPa -> Extract Index: 40 | (Eta: 0.486328, Nominal P: 492.77 hPa)
Target:  600 hPa -> Extract Index: 34 | (Eta: 0.600586, Nominal P: 608.54 hPa)
Target:  700 hPa -> Extract Index: 29 | (Eta: 0.690430, Nominal P: 699.58 hPa)
Target:  850 hPa -> Extract Index: 19 | (Eta: 0.840820, Nominal P: 851.96 hPa)
Target:  925 hPa -> Extract Index: 12 | (Eta: 0.916016, Nominal P: 928.15 h

In [10]:
dsm_subset = dsm.isel(eta=nearest_indices)
dsm_subset

<xarray.Dataset> Size: 51GB
Dimensions:      (time: 124, eta: 13, south_north: 970, west_east: 1389)
Coordinates:
  * time         (time) datetime64[ns] 992B 2019-01-01 ... 2019-01-31T18:00:00
  * eta          (eta) float32 52B 0.04993 0.09753 0.1494 ... 0.916 0.9873
  * south_north  (south_north) float32 4kB -2.827e+03 -2.821e+03 ... 2.988e+03
  * west_east    (west_east) float32 6kB -5.238e+03 -5.232e+03 ... 3.091e+03
    lat          (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
    lon          (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
Data variables: (12/17)
    Z_e          (time, eta, south_north, west_east) float32 9GB dask.array<chunksize=(4, 13, 970, 1389), meta=np.ndarray>
    T_e          (time, eta, south_north, west_east) float32 9GB dask.array<chunksize=(4, 13, 970, 1389), meta=np.ndarray>
    q_e          (time, eta, south_north, west_east) float32 9GB dask.array<chunksize=(4, 13, 970, 1389), meta=np.ndarray>
    u_gr_e       (time, eta, south_north, west_east) float32 9GB dask.array<chunksize=(4, 13, 970, 1389), meta=np.ndarray>
    v_gr_e       (time, eta, south_north, west_east) float32 9GB dask.array<chunksize=(4, 13, 970, 1389), meta=np.ndarray>
    slp          (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    ...           ...
    IVTV         (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    IVTU         (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    u_10m_gr     (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    v_10m_gr     (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    Z_sfc        (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    precip_bkt   (time, south_north, west_east) float32 668MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF 1.6, Standard Name Table v19
    source:         /cw3e/mead/projects/cnt117/reanalysis_temp/input/2018/d01...
    institution:    CW3E - Scripps Institution of Oceanography
    title:          /cw3e/mead/projects/cnt117/reanalysis_temp/output/2018/d0...
    notes:          Created with NCL script:  wrfout_to_cf.ncl v2.0
    created_by:     Daniel Steinhoff - dsteinhoff@ucsd.edu
    creation_date:  Fri Dec  5 01:18:13 PM PST 2025
    history:        Fri Dec  5 13:24:15 2025: /home/cw3eprod/miniforge3/envs/...
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....
    NCO:            netCDF Operators version 5.2.9 (Homepage = http://nco.sf....